# Other model families (issue #17)

Compares logistic regression, random forest, extra trees, k-nearest neighbors, an RBF SVM and a small neural net with the boosting models, using the **shared evaluation protocol** (`src/common.py`, issue #8): development data = train + val, repeated stratified 5-fold CV x 3, seed 42, test set untouched (it is scored once in #18).

Each model is run twice:
1. **Default settings** (scaling inside a Pipeline where the model needs it).
2. **Light tuning with nested CV**: a `RandomizedSearchCV` (few key parameters, 3-fold inner CV) runs inside each outer training fold, so the tuned score is not inflated by tuning. Same method as the boosting notebooks (#10, #12).

Every run is appended to `results/model_results.csv`.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np, pandas as pd
from scipy.stats import loguniform, randint, uniform
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

from src.common import load_dev, evaluate, RANDOM_STATE
warnings.filterwarnings('ignore')

X, y = load_dev()
print(X.shape)

## 1. Default settings (logged)

Distance- and gradient-based models (LR, kNN, SVM, MLP) get a `StandardScaler` inside the pipeline so it is fit on each training fold only. Trees do not need scaling.

In [ ]:
defaults = {
    'Logistic Regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'Random Forest': RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=RANDOM_STATE),
    'Extra Trees': ExtraTreesClassifier(n_estimators=300, n_jobs=1, random_state=RANDOM_STATE),
    'kNN': make_pipeline(StandardScaler(), KNeighborsClassifier()),
    'SVM (RBF)': make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=RANDOM_STATE)),
    'MLP': make_pipeline(StandardScaler(), MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE)),
}
rows = []
for name, model in defaults.items():
    r = evaluate(model, X, y, name=name, params={'defaults': True}, notes='#17 defaults')
    rows.append(r)
    print(f"{name:22s} AUC {r['cv_auc_mean']:.4f} +/- {r['cv_auc_std']:.4f}  fit {r['fit_time_mean_s']}s")
pd.DataFrame(rows)[['model','cv_auc_mean','cv_auc_std','cv_pr_auc_mean','cv_f1_mean','cv_recall_mean','cv_precision_mean','fit_time_mean_s']]

## 2. Light tuning with nested CV (logged)

A few key parameters per model, 12 random draws, 3-fold inner CV on ROC AUC. The inner search runs inside every outer training fold. `n_jobs=1` inside each model because the outer folds already run in parallel. The SVM is the slowest (about 16k rows); lower `N_ITER` if you only want a quick run.

In [ ]:
N_ITER = 12
inner = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)

space = {
    'Logistic Regression': (make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
        {'logisticregression__C': loguniform(1e-3, 1e2)}),
    'Random Forest': (RandomForestClassifier(n_estimators=300, n_jobs=1, random_state=RANDOM_STATE),
        {'max_depth': randint(3, 16), 'min_samples_leaf': randint(5, 80), 'max_features': uniform(0.3, 0.7)}),
    'Extra Trees': (ExtraTreesClassifier(n_estimators=300, n_jobs=1, random_state=RANDOM_STATE),
        {'max_depth': randint(3, 16), 'min_samples_leaf': randint(5, 80), 'max_features': uniform(0.3, 0.7)}),
    'kNN': (make_pipeline(StandardScaler(), KNeighborsClassifier()),
        {'kneighborsclassifier__n_neighbors': randint(15, 301), 'kneighborsclassifier__weights': ['uniform', 'distance']}),
    'SVM (RBF)': (make_pipeline(StandardScaler(), SVC(kernel='rbf', random_state=RANDOM_STATE)),
        {'svc__C': loguniform(0.1, 10), 'svc__gamma': loguniform(1e-3, 1e-1)}),
    'MLP': (make_pipeline(StandardScaler(), MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE)),
        {'mlpclassifier__hidden_layer_sizes': [(16,), (32,), (64,), (32, 16)],
         'mlpclassifier__alpha': loguniform(1e-5, 1e-1),
         'mlpclassifier__learning_rate_init': loguniform(1e-4, 1e-2)}),
}
tuned_rows = []
for name, (est, dist) in space.items():
    search = RandomizedSearchCV(est, dist, n_iter=N_ITER, scoring='roc_auc', cv=inner,
                                random_state=RANDOM_STATE, n_jobs=1)
    r = evaluate(search, X, y, name=f'{name} tuned', params={'search': f'RandomizedSearchCV n_iter={N_ITER}'},
                 notes='#17 NESTED CV (search inside each outer training fold); fit time includes the search')
    tuned_rows.append(r)
    print(f"{name:22s} AUC {r['cv_auc_mean']:.4f} +/- {r['cv_auc_std']:.4f}  fit {r['fit_time_mean_s']}s")
pd.DataFrame(tuned_rows)[['model','cv_auc_mean','cv_auc_std','cv_pr_auc_mean','cv_f1_mean','cv_recall_mean','cv_precision_mean','fit_time_mean_s']]

## 3. Comparison with boosting

In [ ]:
log = pd.read_csv(Path.cwd().parent / 'results' / 'model_results.csv')
cmp = (log[log['notes'].astype(str).str.contains('NESTED') | log['model'].str.contains('Dummy')]
       .sort_values('cv_auc_mean', ascending=False)
       [['model','cv_auc_mean','cv_auc_std','cv_pr_auc_mean','cv_f1_mean','fit_time_mean_s']])
cmp

## 4. Findings

*Fill in after running:* which family comes closest to the boosting models, whether the gap is larger than the fold-to-fold spread (about 0.009 AUC), and fit-time cost. The earlier logistic-regression baseline (PR #5) was about 0.688 AUC.